In [ ]:
import cv2
import os
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import joblib

# --- CONFIGURATION ---
RADIUS = 1
NEIGHBORS = 8
GRID_X = 8
GRID_Y = 8

# Path to your dataset
DATA_PATH = r'..\Dataset\training\Cleaned_Training'

In [5]:
def rotate_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    return rotated

def extract_hog_features(image):
    """
    Extracts Histogram of Oriented Gradients (HOG) features.
    """
    features = hog(image, 
                   orientations=9, 
                   pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), 
                   block_norm='L2-Hys', 
                   visualize=False)
    return features

In [6]:
def load_and_augment_data(path):
    image_paths = [os.path.join(path, f) for f in os.listdir(path)]
    
    # Lists for LBPH (Raw Images)
    lbph_faces = []
    lbph_ids = []
    
    # Lists for SVM & KNN (HOG Feature Vectors)
    hog_features = []
    hog_labels = []
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    print(f"Processing {len(image_paths)} source images...")
    
    for img_path in image_paths:
        try:
            img = Image.open(img_path).convert('L')
            img_np = np.array(img, 'uint8')
            
            # CAUTION: Ensure filenames are "User.ID.jpg"
            user_id = int(os.path.split(img_path)[-1].split(".")[1])
            
            # Base Preprocessing (200x200 standard)
            face_resized = cv2.resize(img_np, (200, 200), interpolation=cv2.INTER_CUBIC)
            face_smooth = cv2.bilateralFilter(face_resized, 5, 75, 75)
            enhanced = clahe.apply(face_smooth)
            
            # --- AUGMENTATION STACK ---
            aug_imgs = [enhanced]
            aug_imgs.append(cv2.flip(enhanced, 1))                        # Flip
            aug_imgs.append(rotate_image(enhanced, -10))                  # Rotate Left
            aug_imgs.append(rotate_image(enhanced, 10))                   # Rotate Right
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=-40)) # Darker
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=40))  # Brighter
            
            # Add to datasets
            for face in aug_imgs:
                # 1. For LBPH: Add the raw image
                lbph_faces.append(face)
                lbph_ids.append(user_id)
                
                # 2. For SVM & KNN: Extract HOG features
                feat = extract_hog_features(face)
                hog_features.append(feat)
                hog_labels.append(user_id)
                
        except Exception as e:
            print(f"Skipping {img_path}: {e}")
            
    return lbph_faces, lbph_ids, hog_features, hog_labels

# --- EXECUTE LOAD ---
faces, ids, hog_feats, hog_lbls = load_and_augment_data(DATA_PATH)

if len(faces) > 0:
    print(f"✅ Data Loaded successfully.")
    print(f"Total Augmented Samples: {len(faces)}")
else:
    print("⚠️ No data found. Check your path.")

Processing 139 source images...
✅ Data Loaded successfully.
Total Augmented Samples: 834


In [ ]:
if len(faces) > 0:
    print("Training LBPH Model (OpenCV)...")
    lbph = cv2.face.LBPHFaceRecognizer_create(radius=RADIUS, neighbors=NEIGHBORS, grid_x=GRID_X, grid_y=GRID_Y)
    lbph.train(faces, np.array(ids))
    lbph.save(r'..\models\trainer.yml')
    print("✅ Saved r'..\models\trainer.yml'")
else:
    print("Skipping LBPH: No data.")

Training LBPH Model (OpenCV)...
✅ Saved 'trainer.yml'


In [ ]:
if len(hog_feats) > 0:
    print("Training SVM Model (HOG)...")
    # SVM Training
    svm = SVC(kernel='linear', C=10.0, gamma='scale', probability=True, random_state=42)
    svm.fit(hog_feats, hog_lbls)
    joblib.dump(svm, r'..\models\svm_face_model.pkl')
    print("✅ Saved r'..\models\svm_face_model.pkl'")
else:
    print("Skipping SVM: No data.")

Training SVM Model (HOG)...
✅ Saved 'svm_face_model.pkl'


In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import joblib
import numpy as np

# Assuming hog_feats and hog_lbls are already populated from your load_and_augment_data function
if len(hog_feats) > 0:
    print("Training XGBoost Model (HOG)...")
    
    # XGBoost requires labels starting from 0
    le = LabelEncoder()
    labels_encoded = le.fit_transform(hog_lbls)
    
    xgb_model = XGBClassifier(
        objective='multi:softprob',
        num_class=len(le.classes_),
        n_estimators=200,
        max_depth=6,
        learning_rate=0.5,
        subsample=0.8,
        colsample_bytree=1,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    )
    
    xgb_model.fit(np.array(hog_feats), labels_encoded)
    
    # Save both the model and the label encoder
    joblib.dump(xgb_model, r'..\models\xgb_face_model.pkl')
    joblib.dump(le, r'..\models\label_encoder.pkl')
    print("✅ Saved 'xgb_face_model.pkl' and 'label_encoder.pkl'")
else:
    print("Skipping XGBoost: No data.")

Training XGBoost Model (HOG)...
✅ Saved 'xgb_face_model.pkl' and 'label_encoder.pkl'
